# Install libraries

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import torch

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [2]:
test_df = pd.read_csv("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
print("testShape:", test_df.shape)
test_df.head()

testShape: (3020, 2)


,id,filename
0,1,mashups/song0001.wav
1,2,mashups/song0002.wav
2,3,mashups/song0003.wav
3,4,mashups/song0004.wav
4,5,mashups/song0005.wav


In [3]:
from transformers import ASTConfig, ASTForAudioClassification, ASTFeatureExtractor

model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)

config = ASTConfig.from_pretrained(model_name)
config.num_labels = 10
model = ASTForAudioClassification.from_pretrained(
    model_name, 
    config=config, 
    ignore_mismatched_sizes=True
)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

model_file = "/kaggle/input/models/uditmaurya1588/newm/pytorch/default/1/Ast.pth"

state_dict = torch.load(model_file, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [5]:
class ASTSubmissionDataset(Dataset):

    def __init__(self, test_csv, mashup_dir, target_w=1024):
        """
        Dataset used for loading test audio files and converting them
        into Mel spectrogram features suitable for AST inference.

        Parameters
        ----------
        test_csv : str
            Path to the test CSV file.
        
        mashup_dir : str
            Directory containing the audio files.

        target_w : int
            Number of time frames expected by the AST model.
        """

        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir
        self.target_w = target_w

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Handle possible differences in column naming
        if "filename" in row:
            filename = row["filename"]
        else:
            filename = f"{str(row['id']).zfill(4)}.wav"

        audio_path = os.path.join(self.mashup_dir, filename)

        # 1. Load Audio
        audio, _ = librosa.load(audio_path, sr=22050, duration=30.0)

        # Ensure consistent audio length (30 seconds)
        expected_length = 22050 * 30

        if len(audio) < expected_length:
            audio = np.pad(audio, (0, expected_length - len(audio)))


        # 2. Convert to Mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=22050,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)


        # 3. Normalize Spectrogram
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)


        # 4. AST Format Conversion
        mel_transposed = mel_db.T


        # 5. Resize to AST Requirement
        if mel_transposed.shape[0] > self.target_w:

            start = (mel_transposed.shape[0] - self.target_w) // 2

            mel_final = mel_transposed[
                start : start + self.target_w, :
            ]

        else:

            mel_final = np.pad(
                mel_transposed,
                ((0, self.target_w - mel_transposed.shape[0]), (0, 0))
            )

        mel_tensor = torch.tensor(mel_final, dtype=torch.float32)

        return mel_tensor, str(row["id"])

In [6]:
# Path to test CSV
test_csv = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"

# Directory containing mashup audio files
mashup_dir = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

# Genre labels used during training
genres = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

# Create dataset
test_dataset = ASTSubmissionDataset(
    test_csv=test_csv,
    mashup_dir=mashup_dir
)

# Create dataloader
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [7]:
model.eval()

predictions = []

with torch.no_grad():

    for mels, ids in tqdm(test_loader, desc="Running Inference"):

        # Move features to device
        mels = mels.to(device)

        # Forward pass
        outputs = model(mels)

        # Handle HuggingFace AST outputs
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs

        # Convert logits to predicted class
        preds = torch.argmax(logits, dim=1)

        preds = preds.cpu().numpy()

        for sample_id, pred_idx in zip(ids, preds):

            predictions.append({
                "id": sample_id.zfill(4),
                "genre": genres[pred_idx]
            })

Running Inference: 100%|██████████| 95/95 [04:19<00:00,  2.73s/it]


In [8]:
submission = pd.DataFrame(predictions)

submission.to_csv("submission.csv", index=False)

print("Submission file saved successfully.")
print("Total predictions:", len(submission))

Submission file saved successfully.
Total predictions: 3020


In [9]:
submission

,id,genre
0,0001,pop
1,0002,classical
2,0003,disco
3,0004,metal
4,0005,country
...,...,...
3015,3016,rock
3016,3017,jazz
3017,3018,reggae
3018,3019,rock
